1. Import for ML

In [1]:
!pip install pandas scikit-learn sentence-transformers

  Using cached scikit_learn-1.7.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (11 kB)
  Using cached sentence_transformers-6.0.0-py3-none-any.whl.metadata (20 kB)
  Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached transformers-5.15.1-py3-none-any.whl.metadata (32 kB)
  Using cached tokenizers-0.23.1-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (9.8 kB)
  Using cached huggingface_hub-1.28.0-py3-none-any.whl.metadata (16 kB)
  Using cached torch-2.13.0-cp310-cp310-manylinux_2_28_x86_64.whl.metadata (38 kB)
  Using cached filelock-3.32.4-py3-none-any.whl.metadata (2.0 kB)
  Using cached hf_xet-1.6.0-cp38-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (4.9 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cach

In [2]:
import pandas as pd
import numpy as np
import re
from difflib import SequenceMatcher
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, precision_recall_fscore_support, accuracy_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import hstack, csr_matrix
from sentence_transformers import SentenceTransformer

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2. Load pairwise gold dataset with Bin

In [3]:
# Load the dataset with fixed Bin assignments
df = pd.read_csv(
    "pairwise_gold_dataset_with_bins.csv",
    encoding="utf-8-sig"
)

print(pd.crosstab(df["Bin"], df["Label"]))

Label    0   1
Bin           
0      283  57
1      283  57
2      283  57
3      283  57
4      283  57
5      283  57
6      283  56
7      283  56
8      283  56
9      283  56


3. Text normalization helpers

In [4]:
def normalize_text(text):
    # Convert text to lowercase and remove punctuation
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text):
    # Tokenize normalized text
    return normalize_text(text).split()


def acronym(text):
    # Create acronym from organization name tokens
    tokens = tokenize(text)
    return "".join(t[0] for t in tokens if t)


def jaccard_similarity(text1, text2):
    # Compute token-level Jaccard similarity
    s1 = set(tokenize(text1))
    s2 = set(tokenize(text2))
    
    if len(s1) == 0 and len(s2) == 0:
        return 0
    
    return len(s1 & s2) / len(s1 | s2)


def sequence_similarity(text1, text2):
    # Compute character-level sequence similarity
    return SequenceMatcher(None, normalize_text(text1), normalize_text(text2)).ratio()


def acronym_match(text1, text2):
    # Check whether acronyms match
    a1 = acronym(text1)
    a2 = acronym(text2)
    
    if len(a1) < 2 or len(a2) < 2:
        return 0
    
    return int(a1 == a2)


def length_features(text1, text2):
    # Compute simple length-based features
    n1 = len(normalize_text(text1))
    n2 = len(normalize_text(text2))
    
    return {
        "len_1": n1,
        "len_2": n2,
        "len_diff": abs(n1 - n2),
        "len_ratio": min(n1, n2) / max(n1, n2) if max(n1, n2) > 0 else 0
    }

4. Numeric similarity features

In [5]:
def build_numeric_features(dataframe):
    # Build hand-crafted numeric features
    rows = []
    
    for _, row in dataframe.iterrows():
        name1 = row["OrgName_1"]
        name2 = row["OrgName_2"]
        
        features = {
            "jaccard": jaccard_similarity(name1, name2),
            "sequence_similarity": sequence_similarity(name1, name2),
            "acronym_match": acronym_match(name1, name2)
        }
        
        features.update(length_features(name1, name2))
        rows.append(features)
    
    return pd.DataFrame(rows)

5. Embedding cosine similarity feature

In [6]:
# Load multilingual embedding model
embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# Encode all unique organization names once
unique_names = pd.unique(
    pd.concat([df["OrgName_1"], df["OrgName_2"]], ignore_index=True)
)

name_embeddings = embedding_model.encode(
    unique_names.tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

embedding_dict = {
    name: emb for name, emb in zip(unique_names, name_embeddings)
}


def get_embedding_cosine(name1, name2):
    # Compute cosine similarity between two precomputed embeddings
    emb1 = embedding_dict[name1]
    emb2 = embedding_dict[name2]
    return float(np.dot(emb1, emb2))


df["embedding_cosine"] = df.apply(
    lambda row: get_embedding_cosine(row["OrgName_1"], row["OrgName_2"]),
    axis=1
)

Batches: 100%|██████████| 20/20 [00:02<00:00,  8.26it/s]


6. Feature builder for each fold

In [7]:
def build_features_for_fold(train_df, test_df):
    # Combine two organization names as pair text for n-gram features
    train_pair_text = (
        train_df["OrgName_1"].astype(str) 
        + " [SEP] " 
        + train_df["OrgName_2"].astype(str)
    )
    
    test_pair_text = (
        test_df["OrgName_1"].astype(str) 
        + " [SEP] " 
        + test_df["OrgName_2"].astype(str)
    )
    
    # Character n-gram TF-IDF features
    vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(2, 5),
        min_df=1
    )
    
    X_train_ngram = vectorizer.fit_transform(train_pair_text)
    X_test_ngram = vectorizer.transform(test_pair_text)
    
    # Numeric features
    train_numeric = build_numeric_features(train_df)
    test_numeric = build_numeric_features(test_df)
    
    # Add embedding cosine similarity
    train_numeric["embedding_cosine"] = train_df["embedding_cosine"].values
    test_numeric["embedding_cosine"] = test_df["embedding_cosine"].values
    
    X_train_numeric = csr_matrix(train_numeric.values)
    X_test_numeric = csr_matrix(test_numeric.values)
    
    # Combine n-gram and numeric features
    X_train = hstack([X_train_ngram, X_train_numeric])
    X_test = hstack([X_test_ngram, X_test_numeric])
    
    y_train = train_df["Label"].values
    y_test = test_df["Label"].values
    
    return X_train, X_test, y_train, y_test

7. 10-bin evaluation: 8 train, 1 test, 1 dev

In [8]:
from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score

def run_10bin_evaluation_with_pair_outputs(df):
    # Store fold-level evaluation results
    fold_results = []
    
    # Store pair-level prediction results
    pair_predictions = []
    
    for test_bin in range(10):
        # Use the next bin as development set
        dev_bin = (test_bin + 1) % 10
        
        train_bins = [
            b for b in range(10)
            if b not in [test_bin, dev_bin]
        ]
        
        train_df = df[df["Bin"].isin(train_bins)].reset_index(drop=True)
        dev_df = df[df["Bin"] == dev_bin].reset_index(drop=True)
        test_df = df[df["Bin"] == test_bin].reset_index(drop=True)
        
        print(f"\nFold {test_bin}")
        print("Train:", train_df.shape, train_df["Label"].value_counts().to_dict())
        print("Dev:", dev_df.shape, dev_df["Label"].value_counts().to_dict())
        print("Test:", test_df.shape, test_df["Label"].value_counts().to_dict())
        
        X_train, X_test, y_train, y_test = build_features_for_fold(train_df, test_df)
        
        models = {
            "LogisticRegression": LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=RANDOM_SEED
            ),
            "RandomForest": RandomForestClassifier(
                n_estimators=100,
                max_depth=50,
                min_samples_leaf=2,
                class_weight="balanced",
                random_state=RANDOM_SEED,
                n_jobs=-1
            )
        }
        
        for model_name, model in models.items():
            model.fit(X_train, y_train)
            
            # Predicted class labels
            y_pred = model.predict(X_test)
            
            # Probability score for the positive class
            y_prob = model.predict_proba(X_test)[:, 1]
            
            # Overall metrics
            overall_precision, overall_recall, overall_f1, _ = precision_recall_fscore_support(
                y_test,
                y_pred,
                average="binary",
                pos_label=1,
                zero_division=0
            )
            
            # Class-specific metrics: class 0 = non-match, class 1 = match
            class_precision, class_recall, class_f1, class_support = precision_recall_fscore_support(
                y_test,
                y_pred,
                labels=[0, 1],
                average=None,
                zero_division=0
            )
            
            auc = roc_auc_score(y_test, y_prob)
            acc = accuracy_score(y_test, y_pred)
            
            # Save fold-level results
            fold_results.append({
                "Fold": test_bin,
                "Model": model_name,
                "Accuracy": acc,
                "ROC_AUC": auc,
                
                "Match_Precision": class_precision[1],
                "Match_Recall": class_recall[1],
                "Match_F1": class_f1[1],
                "Match_Support": class_support[1],
                
                "NonMatch_Precision": class_precision[0],
                "NonMatch_Recall": class_recall[0],
                "NonMatch_F1": class_f1[0],
                "NonMatch_Support": class_support[0],
                
                "Overall_PositiveClass_Precision": overall_precision,
                "Overall_PositiveClass_Recall": overall_recall,
                "Overall_PositiveClass_F1": overall_f1
            })
            
            # Save pair-level predictions
            temp_pred_df = test_df.copy()
            temp_pred_df["Fold"] = test_bin
            temp_pred_df["Model"] = model_name
            temp_pred_df["TrueLabel"] = y_test
            temp_pred_df["PredictedLabel"] = y_pred
            temp_pred_df["Probability_Match"] = y_prob
            temp_pred_df["Probability_NonMatch"] = 1 - y_prob
            
            pair_predictions.append(temp_pred_df)
            
            print("\n", model_name)
            print(classification_report(
                y_test,
                y_pred,
                target_names=["NonMatch", "Match"],
                digits=4,
                zero_division=0
            ))
    
    fold_results_df = pd.DataFrame(fold_results)
    pair_predictions_df = pd.concat(pair_predictions, ignore_index=True)
    
    return fold_results_df, pair_predictions_df

8. Run

In [9]:
fold_results_df, pair_predictions_df = run_10bin_evaluation_with_pair_outputs(df)


Fold 0
Train: (2716, 10) {0: 2264, 1: 452}
Dev: (340, 10) {0: 283, 1: 57}
Test: (340, 10) {0: 283, 1: 57}

 LogisticRegression
              precision    recall  f1-score   support

    NonMatch     0.9857    0.9717    0.9786       283
       Match     0.8689    0.9298    0.8983        57

    accuracy                         0.9647       340
   macro avg     0.9273    0.9508    0.9385       340
weighted avg     0.9661    0.9647    0.9652       340


 RandomForest
              precision    recall  f1-score   support

    NonMatch     0.9625    0.9965    0.9792       283
       Match     0.9787    0.8070    0.8846        57

    accuracy                         0.9647       340
   macro avg     0.9706    0.9017    0.9319       340
weighted avg     0.9652    0.9647    0.9633       340


Fold 1
Train: (2716, 10) {0: 2264, 1: 452}
Dev: (340, 10) {0: 283, 1: 57}
Test: (340, 10) {0: 283, 1: 57}

 LogisticRegression
              precision    recall  f1-score   support

    NonMatch     0.9

9. Save

In [10]:
# Save fold-level evaluation results
fold_results_df.to_csv(
    "ml_fold_level_results_match_nonmatch.csv",
    index=False,
    encoding="utf-8-sig"
)

# Save pair-level prediction results
pair_predictions_df.to_csv(
    "ml_pair_level_predictions_with_probability.csv",
    index=False,
    encoding="utf-8-sig"
)

10. Summary

In [11]:
summary_df = (
    fold_results_df
    .groupby("Model")[
        [
            "Match_Precision",
            "Match_Recall",
            "Match_F1",
            "NonMatch_Precision",
            "NonMatch_Recall",
            "NonMatch_F1",
            "ROC_AUC"
        ]
    ]
    .agg(["mean", "std"])
)

print(summary_df)

summary_df.to_csv(
    "ml_summary_match_nonmatch.csv",
    encoding="utf-8-sig"
)

                   Match_Precision           Match_Recall            Match_F1  \
                              mean       std         mean       std      mean   
Model                                                                           
LogisticRegression        0.817405  0.040704     0.913283  0.035409  0.861917   
RandomForest              0.966585  0.024401     0.748653  0.066918  0.842087   

                             NonMatch_Precision           NonMatch_Recall  \
                         std               mean       std            mean   
Model                                                                       
LogisticRegression  0.027784           0.982313  0.006886        0.958657   
RandomForest        0.042104           0.952123  0.011845        0.994700   

                             NonMatch_F1             ROC_AUC            
                         std        mean       std      mean       std  
Model                                                         

11. Score for Clustering

In [11]:
hac_score_df = pair_predictions_df[
    [
        "Model",
        "Fold",
        "InstanceID_1",
        "OrgName_1",
        "InstanceID_2",
        "OrgName_2",
        "TrueLabel",
        "PredictedLabel",
        "Probability_Match"
    ]
].copy()

hac_score_df = hac_score_df.rename(
    columns={"Probability_Match": "SimilarityScore"}
)

hac_score_df.to_csv(
    "hac_similarity_scores_from_ml.csv",
    index=False,
    encoding="utf-8-sig"
)